# Learn-a-World Phase 2: Dual-Store Online Causal Memory Screening (Arm C)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/main/notebooks/06_screen_dual_store_colab.ipynb)

This notebook executes the **50-episode official screening (seeds 2050–2099)** on GPU for **Arm C: Dual-Store Online Causal Memory**.

### Key Architecture Highlights:
- **Neocortical Semantic Core + Quarantined Sandbox**: Hypotheses are isolated until verified.
- **Physical Invariance Gate**: Zero spurious hypotheses from sensor noise.
- **Bayesian Falsification Pruning**: Non-productive hypotheses are evicted; rare milestones are protected.
- **Active-Link Topological Gating**: Eliminates premature execution at crafting stations.
- **Zero-Hardcoding Guarantee**: 100% compliant with the strict sensory protocol (RGB + scalar reward only).


### Step 1: Mount Google Drive
Mounts Google Drive to load checkpoints and mirror screening results.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/learn-a-world")
(DRIVE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / "results").mkdir(parents=True, exist_ok=True)

print("Google Drive workspace ready at:", DRIVE_DIR)

### Step 2: Clone or Pull Repository on Branch 

In [ ]:
%%bash
cd /content
if [ ! -d "learn-a-world" ]; then
  git clone https://github.com/sahiltolani30/learn-a-world.git
fi
cd learn-a-world
git fetch origin
git checkout main
git pull origin main
echo "--> Latest commit:"
git log -1 --oneline

### Step 3: Install Dependencies & Check GPU

In [ ]:
%%bash
cd /content/learn-a-world
pip install -q -r requirements.txt
pip install -q crafter gymnasium pytest
nvidia-smi

### Step 4: Run Pre-Flight Automated Unit & Anti-Hardcoding Tests

In [ ]:
%%bash
cd /content/learn-a-world
python3 scripts/strict_anti_hardcoding_audit.py
pytest tests/test_phase2_*.py tests/test_dual_store_backward_compatibility.py -v

### Step 5: Verify & Copy Checkpoint / Bank Assets

In [ ]:
import shutil
from pathlib import Path

# 1. Model Checkpoint
drive_ckpt = Path("/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt")
local_ckpt = Path("/content/learn-a-world/checkpoints/jepa_crafter_v3_phase2.pt")
local_ckpt.parent.mkdir(parents=True, exist_ok=True)

if drive_ckpt.exists() and not local_ckpt.exists():
    shutil.copy(drive_ckpt, local_ckpt)
    print(f"Copied {drive_ckpt} -> {local_ckpt}")
elif local_ckpt.exists():
    print(f"Local checkpoint ready: {local_ckpt} ({local_ckpt.stat().st_size / 1e6:.1f} MB)")
else:
    print("Notice: Checkpoint not found in Drive. Make sure jepa_crafter_v3_phase2.pt is present.")

# 2. Base Causal Memory Bank
drive_bank = Path("/content/drive/MyDrive/learn-a-world/checkpoints/phase2_causal_memory_bank_v2.json")
local_bank = Path("/content/learn-a-world/checkpoints/phase2_causal_memory_bank_v2.json")
if drive_bank.exists() and not local_bank.exists():
    shutil.copy(drive_bank, local_bank)
    print(f"Copied {drive_bank} -> {local_bank}")
elif local_bank.exists():
    print(f"Local bank ready: {local_bank} ({local_bank.stat().st_size / 1e6:.1f} MB)")
else:
    print("Warning: phase2_causal_memory_bank_v2.json not found in Drive or local workspace!")


### Step 6: Launch 50-Episode GPU Screening (Seeds 2050–2099)
Executes Recurrent MPC with online dual-store learning on GPU, mirroring results directly to Google Drive.

In [ ]:
!cd /content/learn-a-world && python3 scripts/screen_dual_store_50eps.py \n  --checkpoint checkpoints/jepa_crafter_v3_state_readout_v2.pt \n  --causal_memory_bank phase2_memory_consolidation_4.json \n  --online_learning \n  --online_bank_path results/phase2_causal_memory_bank_online.json \n  --output_json results/screen_v3_armC_dual_store_50ep.json \n  --use_transition_model \n  --transition_scoring \n  --death_risk_factor 0.5 \n  --horizon 15 \n  --num_candidates 64 \n  --cem_iterations 2 \n  --device cuda \n  --drive_dir /content/drive/MyDrive/learn-a-world


### Step 7: View Summary & Compare Against Baseline

In [ ]:
import json
from pathlib import Path

res_file = Path("/content/learn-a-world/results/screen_v3_armC_dual_store_50ep.json")
if not res_file.exists():
    res_file = Path("/content/drive/MyDrive/learn-a-world/results/screen_v3_armC_dual_store_50ep.json")

if res_file.exists():
    with open(res_file) as f:
        data = json.load(f)
    print(f"Official Crafter Score: {data.get('crafter_score', 0):.2f}%")
    print(f"Mean Episode Reward: {data.get('mean_episode_reward', 0):.2f}")
    print(f"Mean Survival Length: {data.get('mean_survival_length', 0):.1f}")
    print("
Unlock Percentages:")
    for k, v in data.get("unlock_percentages", {}).items():
        if v > 0:
            print(f"  {k:22s}: {v:.1f}%")
else:
    print("Results file not found.")